# 04 · Reshape and transpose real images / Reshape y transposición de imágenes reales

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/project-delphi/tensors-workshop/blob/main/notebooks/04-reshape-and-transpose.ipynb)

*Part III · exercise · 15 min*

This notebook teaches one idea that prevents many silent bugs:

> **Changing a tensor's shape is not the same as moving its axes.**

We will use real images to see when `transpose` is the correct operation and why `reshape` can produce the expected numbers in `.shape` while giving the wrong interpretation.

> 🇪🇸 Este cuaderno enseña una idea que evita muchos errores silenciosos:
>
> **Cambiar la forma de un tensor no es lo mismo que mover sus ejes.**
>
> Usaremos imágenes reales para entender cuándo `transpose` es la operación correcta y por qué `reshape` puede producir la `.shape` esperada y, aun así, dar una interpretación equivocada.

## What you will be able to do / Lo que podrás hacer

- Read `HWC`, `CHW`, `NHWC`, and `NCHW` as simple sentences.
- Convert a real RGB image from `(H, W, C)` to `(C, H, W)` with `np.transpose`.
- Build a real batch from three different RGB images and convert `NHWC → NCHW`.
- Explain why two axes can have the same size but different meanings.
- Demonstrate visually why `reshape` cannot replace `transpose` when axis meaning must move.

> 🇪🇸
>
> - Leer `HWC`, `CHW`, `NHWC` y `NCHW` como frases sencillas.
> - Convertir una imagen RGB real de `(H, W, C)` a `(C, H, W)` con `np.transpose`.
> - Construir un lote real con tres imágenes RGB distintas y convertir `NHWC → NCHW`.
> - Explicar por qué dos ejes pueden tener el mismo tamaño y significados diferentes.
> - Demostrar visualmente por qué `reshape` no puede sustituir `transpose` cuando debe cambiar la posición de los ejes.

## Start with a simple analogy / Empecemos con una analogía sencilla

Imagine three labeled drawers:

`[Height, Width, Colour]`

A **transpose** operation is like physically moving the drawers:

`[Height, Width, Colour] → [Colour, Height, Width]`

The contents stay associated with the correct drawer; only the drawer positions change.

A **reshape** operation is different. It keeps reading the stored values in their existing memory order and groups them into a new shape.

That is why:

**same output shape ≠ same data meaning**

> 🇪🇸 Imagina tres cajones con etiquetas:
>
> `[Alto, Ancho, Color]`
>
> Una **transposición** mueve físicamente los cajones:
>
> `[Alto, Ancho, Color] → [Color, Alto, Ancho]`
>
> El contenido sigue asociado con el eje correcto; únicamente cambia la posición.
>
> **Reshape** hace otra cosa: conserva el orden de los valores almacenados y los vuelve a agrupar con una forma nueva.
>
> Por eso:
>
> **misma forma de salida ≠ mismo significado de los datos**

## Four axis letters / Cuatro letras de ejes

| Letter / Letra | English | Español |
|---|---|---|
| `N` | number of examples / batch | número de ejemplos / lote |
| `H` | height | alto |
| `W` | width | ancho |
| `C` | colour channels | canales de color |

Read the conventions as sentences:

- `HWC` = height × width × colour
- `CHW` = colour × height × width
- `NHWC` = examples × height × width × colour
- `NCHW` = examples × colour × height × width

> 🇪🇸 Lee las convenciones como frases:
>
> - `HWC` = alto × ancho × color
> - `CHW` = color × alto × ancho
> - `NHWC` = ejemplos × alto × ancho × color
> - `NCHW` = ejemplos × color × alto × ancho

## Setup / Preparación

We will use real images distributed with `scikit-image`:

- histology,
- microscopy,
- astronaut photograph,
- coffee photograph.

No synthetic image pixels are needed for the exercises.

> 🇪🇸 Usaremos imágenes reales incluidas en `scikit-image`:
>
> - histología,
> - microscopía,
> - fotografía de astronauta,
> - fotografía de café.
>
> Los ejercicios no necesitan píxeles de imágenes sintéticas.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets

from IPython.display import display
from skimage import data

try:
    from google.colab import output
    output.enable_custom_widget_manager()
except ImportError:
    pass

# Real images distributed with scikit-image.
photo = data.immunohistochemistry()   # (512, 512, 3), RGB histology
cells_img = data.cell()               # (660, 550), grayscale microscopy
astronaut = data.astronaut()          # (512, 512, 3), RGB photograph
coffee = data.coffee()                # (400, 600, 3), RGB photograph

def center_crop_rgb(img, size=256):
    # Deterministic centre crop so different real RGB images can be stacked.
    h, w, c = img.shape

    if c != 3 or h < size or w < size:
        raise ValueError(
            f"expected RGB image at least {size}x{size}, got {img.shape}"
        )

    r0 = (h - size) // 2
    c0 = (w - size) // 2

    return img[r0:r0 + size, c0:c0 + size]

rgb_sources = [photo, astronaut, coffee]
rgb_names = [
    "Histology / Histología",
    "Astronaut / Astronauta",
    "Coffee / Café",
]

print("Histology / Histología:", photo.shape)
print("Microscopy / Microscopía:", cells_img.shape)
print("Astronaut / Astronauta:", astronaut.shape)
print("Coffee / Café:", coffee.shape)
print()
print("EN: Setup ready.")
print("ES: Preparación lista.")

## Why this matters / Por qué esto importa

Different software systems may expect image axes in different orders.

For example, one system may use:

`NHWC`

while another expects:

`NCHW`

The tensor may contain all the correct numbers, but a model can still interpret them incorrectly if the axes are in the wrong positions.

The difficult part is that this mistake may **not produce an error message**.

The code can run while the data meaning is wrong.

### Learning habit / Hábito de aprendizaje

Before changing a tensor:

**Predict → Run → Explain**

1. Write the current axis order.
2. Write the desired axis order.
3. Predict the new shape.
4. Run the operation.
5. Explain what each output axis means.

> 🇪🇸 Distintos sistemas pueden esperar los ejes de las imágenes en órdenes diferentes.
>
> Un tensor puede contener todos los números correctos y aun así ser interpretado incorrectamente si los ejes están en posiciones equivocadas.
>
> Lo peligroso es que este error puede **no producir ningún mensaje de error**.
>
> Antes de cambiar un tensor: **Predice → Ejecuta → Explica**.

### Interactive convention translator / Traductor interactivo de convenciones

Choose a convention and read what every position means.

> 🇪🇸 Elige una convención y observa qué significa cada posición.

In [ ]:
convention = widgets.ToggleButtons(
    options=["HWC", "CHW", "NHWC", "NCHW"],
    value="HWC",
    description="Convention / Convención:",
    style={"description_width": "145px"},
)

def explain_convention(value):
    meanings = {
        "HWC": (
            "(H, W, C)",
            "height × width × colour",
            "alto × ancho × color",
        ),
        "CHW": (
            "(C, H, W)",
            "colour × height × width",
            "color × alto × ancho",
        ),
        "NHWC": (
            "(N, H, W, C)",
            "examples × height × width × colour",
            "ejemplos × alto × ancho × color",
        ),
        "NCHW": (
            "(N, C, H, W)",
            "examples × colour × height × width",
            "ejemplos × color × alto × ancho",
        ),
    }

    symbols, en, es = meanings[value]

    print("Symbols / Símbolos:", symbols)
    print("EN:", en)
    print("ES:", es)

convention_output = widgets.interactive_output(
    explain_convention,
    {"value": convention},
)

display(widgets.VBox([convention, convention_output]))

## Exercise 1 — one real image, two orderings / Ejercicio 1 — una imagen real, dos órdenes

`photo` is a real RGB histology image:

`(H, W, C) = (512, 512, 3)`

`cells_img` is a real grayscale microscopy image:

`(H, W) = (660, 550)`

### Predict first / Predice primero

1. Which image has a colour axis?
2. What should `photo.shape` become after `HWC → CHW`?
3. Which old axis should become the new axis 0?

> 🇪🇸
>
> 1. ¿Cuál imagen tiene eje de color?
> 2. ¿Qué forma debería tener `photo` después de `HWC → CHW`?
> 3. ¿Qué eje antiguo debe convertirse en el nuevo eje 0?

In [ ]:
# TODO 1 / TAREA 1
#
# EN:
# 1. Print photo.shape and cells_img.shape.
# 2. Explain why cells_img has no colour axis.
#
# ES:
# 1. Imprime photo.shape y cells_img.shape.
# 2. Explica por qué cells_img no tiene eje de color.
#
# TODO 2 / TAREA 2
#
# EN:
# Convert photo from (H, W, C) to (C, H, W) with np.transpose.
# Name every output axis.
#
# ES:
# Convierte photo de (H, W, C) a (C, H, W) con np.transpose.
# Nombra cada eje de salida.

In [ ]:
#@title Solution / Solución — try it yourself first / inténtalo primero { display-mode: 'form' }

print("photo / foto:", photo.shape)
print("cells_img / microscopía:", cells_img.shape)
print()

print("EN: photo has three axes: height, width, colour.")
print("ES: photo tiene tres ejes: alto, ancho y color.")
print("EN: cells_img is grayscale, so it stores only height and width.")
print("ES: cells_img es en escala de grises, por eso solo almacena alto y ancho.")
print()

chw = np.transpose(photo, (2, 0, 1))

print("HWC:", photo.shape)
print("CHW:", chw.shape)
print()
print("EN: new axis 0 <- old axis 2 (colour)")
print("ES: nuevo eje 0 <- eje antiguo 2 (color)")
print("EN: new axis 1 <- old axis 0 (height)")
print("ES: nuevo eje 1 <- eje antiguo 0 (alto)")
print("EN: new axis 2 <- old axis 1 (width)")
print("ES: nuevo eje 2 <- eje antiguo 1 (ancho)")

assert np.array_equal(chw, np.moveaxis(photo, 2, 0))

### Interactive HWC ↔ CHW explorer / Explorador interactivo HWC ↔ CHW

Choose a colour channel.

The left panel reads the data in the original `HWC` representation.

The right panel reads the **same measured channel** from the transposed `CHW` representation.

If the transpose was correct, the two images must match exactly.

> 🇪🇸 Elige un canal de color.
>
> El panel izquierdo lee los datos desde la representación original `HWC`.
>
> El panel derecho lee **el mismo canal medido** desde la representación transpuesta `CHW`.
>
> Si la transposición es correcta, ambas imágenes deben coincidir exactamente.

In [ ]:
# The HWC->CHW transpose (Exercise 1, TODO 2), recomputed here so this
# explorer runs whether or not the folded solution was executed.
chw = np.transpose(photo, (2, 0, 1))

channel_selector = widgets.ToggleButtons(
    options=[
        ("R · Red / Rojo", 0),
        ("G · Green / Verde", 1),
        ("B · Blue / Azul", 2),
    ],
    value=0,
    description="Channel / Canal:",
    style={"description_width": "120px"},
)

def compare_hwc_chw(channel):
    channel_names = {
        0: "Red / Rojo",
        1: "Green / Verde",
        2: "Blue / Azul",
    }

    from_hwc = photo[:, :, channel]
    from_chw = chw[channel]

    plt.close("all")
    fig, axes = plt.subplots(1, 2, figsize=(8.5, 3.8))

    axes[0].imshow(from_hwc, cmap="gray")
    axes[0].set_title(
        f"HWC → photo[:, :, {channel}]\n{channel_names[channel]}"
    )

    axes[1].imshow(from_chw, cmap="gray")
    axes[1].set_title(
        f"CHW → chw[{channel}]\n{channel_names[channel]}"
    )

    for ax in axes:
        ax.axis("off")

    plt.tight_layout()
    plt.show()

    print(
        "Same measured values / Mismos valores medidos:",
        np.array_equal(from_hwc, from_chw),
    )
    print("EN: transpose moved the colour axis; it did not change the pixel values.")
    print("ES: transpose movió el eje de color; no cambió los valores de los píxeles.")

channel_output = widgets.interactive_output(
    compare_hwc_chw,
    {"channel": channel_selector},
)

display(widgets.VBox([channel_selector, channel_output]))

<details>
<summary><strong>Why this solution works / Por qué funciona esta solución</strong></summary>

`np.transpose(photo, (2, 0, 1))` does not mean “make a shape `(3,512,512)`.”

It means:

**put old axis 2 first, then old axis 0, then old axis 1.**

Because the original axes were:

`(H, W, C)`

the new order becomes:

`(C, H, W)`

> 🇪🇸 `np.transpose(photo, (2, 0, 1))` no significa simplemente “crear una forma `(3,512,512)`”.
>
> Significa:
>
> **coloca primero el eje antiguo 2, después el eje antiguo 0 y finalmente el eje antiguo 1.**
>
> Como el orden original era `(H,W,C)`, el nuevo orden es `(C,H,W)`.

</details>

## 4.2 Add a batch axis / Agrega un eje de lote

A single RGB image can be:

`(H, W, C)`

A batch of RGB images adds one new axis:

`(N, H, W, C)`

where `N` tells us **which image**.

We will build a real batch from:

1. histology,
2. astronaut,
3. coffee.

The images have different original spatial sizes, so we take a deterministic `256 × 256` centre crop from each one before stacking them.

No pixel values are invented.

> 🇪🇸 Una sola imagen RGB puede representarse como `(H,W,C)`.
>
> Un lote agrega un nuevo eje:
>
> `(N,H,W,C)`
>
> donde `N` indica **qué imagen**.
>
> Usaremos histología, astronauta y café. Como sus tamaños originales son diferentes, tomamos un recorte central real de `256 × 256` de cada una antes de apilarlas.

## Exercise 2 — a real batch and two axes of size 3 / Ejercicio 2 — un lote real y dos ejes de tamaño 3

After stacking the three RGB images:

`NHWC = (3, 256, 256, 3)`

After transposing:

`NCHW = (3, 3, 256, 256)`

Notice something important:

**two axes now have size 3.**

But:

- axis 0 = three different images;
- axis 1 = three colour channels.

The number `3` alone does not tell us the meaning.

> 🇪🇸 Después de la transposición obtenemos `(3,3,256,256)`.
>
> Dos ejes tienen tamaño `3`, pero uno representa **tres imágenes** y el otro **tres canales de color**.
>
> El número por sí solo no contiene el significado.

In [ ]:
# TODO 3 / TAREA 3
#
# EN:
# 1. Centre-crop every image in rgb_sources to 256×256.
# 2. Stack them into one batch.
# 3. Verify shape (3, 256, 256, 3).
# 4. Explain the meaning of N, H, W, and C.
#
# ES:
# 1. Recorta al centro cada imagen de rgb_sources a 256×256.
# 2. Apílalas en un solo lote.
# 3. Verifica la forma (3, 256, 256, 3).
# 4. Explica el significado de N, H, W y C.
#
# TODO 4 / TAREA 4
#
# EN:
# Convert NHWC to NCHW.
# Explain why axis 0 and axis 1 both have size 3 but different meanings.
#
# ES:
# Convierte NHWC a NCHW.
# Explica por qué los ejes 0 y 1 tienen tamaño 3 pero significados diferentes.

In [ ]:
#@title Solution / Solución — try it yourself first / inténtalo primero { display-mode: 'form' }

batch = np.stack([center_crop_rgb(img) for img in rgb_sources])

print("NHWC:", batch.shape)
print("EN: N=3 images, H=256, W=256, C=3 colour channels.")
print("ES: N=3 imágenes, H=256, W=256, C=3 canales de color.")
print()

nchw = np.transpose(batch, (0, 3, 1, 2))

print("NCHW:", nchw.shape)
print("EN: axis 0 = image; axis 1 = colour.")
print("ES: eje 0 = imagen; eje 1 = color.")
print()

one_image = nchw[0]
red_all_images = nchw[:, 0]

print("nchw[0].shape:", one_image.shape)
print("EN: one image, all three channels.")
print("ES: una imagen, sus tres canales.")
print()

print("nchw[:, 0].shape:", red_all_images.shape)
print("EN: red channel from all three images.")
print("ES: canal rojo de las tres imágenes.")

assert batch.shape == (3, 256, 256, 3)
assert nchw.shape == (3, 3, 256, 256)
assert np.array_equal(
    one_image,
    np.transpose(batch[0], (2, 0, 1)),
)
assert np.array_equal(
    red_all_images,
    batch[:, :, :, 0],
)

### Interactive batch-axis explorer / Explorador interactivo de ejes del lote

Choose:

- an image `N`,
- a colour channel `C`.

The notebook will read the selected data from both `NHWC` and `NCHW`.

This is useful because both `N` and `C` have size `3`, but the controls make their roles explicit.

> 🇪🇸 Elige:
>
> - una imagen `N`,
> - un canal de color `C`.
>
> El cuaderno leerá los mismos datos desde `NHWC` y `NCHW`.
>
> Esto ayuda a ver que `N` y `C` pueden tener el mismo tamaño `3` y, aun así, representar cosas diferentes.

In [ ]:
# The real batch and its NHWC->NCHW transpose (Exercise 2), recomputed here
# so this explorer runs whether or not the folded solution was executed.
batch = np.stack([center_crop_rgb(img) for img in rgb_sources])
nchw = np.transpose(batch, (0, 3, 1, 2))

image_selector = widgets.ToggleButtons(
    options=[
        ("N=0 · Histology / Histología", 0),
        ("N=1 · Astronaut / Astronauta", 1),
        ("N=2 · Coffee / Café", 2),
    ],
    value=0,
    description="Image N / Imagen N:",
    style={"description_width": "130px"},
)

batch_channel_selector = widgets.ToggleButtons(
    options=[
        ("C=0 · Red / Rojo", 0),
        ("C=1 · Green / Verde", 1),
        ("C=2 · Blue / Azul", 2),
    ],
    value=0,
    description="Channel C / Canal C:",
    style={"description_width": "130px"},
)

def explore_batch_axes(image_idx, channel_idx):
    from_nhwc = batch[image_idx, :, :, channel_idx]
    from_nchw = nchw[image_idx, channel_idx, :, :]

    plt.close("all")
    fig, axes = plt.subplots(1, 3, figsize=(10.5, 3.4))

    axes[0].imshow(batch[image_idx])
    axes[0].set_title(
        f"N={image_idx}\n{rgb_names[image_idx]}"
    )

    axes[1].imshow(from_nhwc, cmap="gray")
    axes[1].set_title(
        f"NHWC[{image_idx}, :, :, {channel_idx}]"
    )

    axes[2].imshow(from_nchw, cmap="gray")
    axes[2].set_title(
        f"NCHW[{image_idx}, {channel_idx}, :, :]"
    )

    for ax in axes:
        ax.axis("off")

    plt.tight_layout()
    plt.show()

    print(
        "Same measured plane / Mismo plano medido:",
        np.array_equal(from_nhwc, from_nchw),
    )
    print(f"N={image_idx} -> EN: which image | ES: qué imagen")
    print(f"C={channel_idx} -> EN: which colour | ES: qué color")

batch_output = widgets.interactive_output(
    explore_batch_axes,
    {
        "image_idx": image_selector,
        "channel_idx": batch_channel_selector,
    },
)

display(
    widgets.VBox([
        image_selector,
        batch_channel_selector,
        batch_output,
    ])
)

<details>
<summary><strong>Why two size-3 axes are different / Por qué dos ejes de tamaño 3 son diferentes</strong></summary>

In:

`(N, C, H, W) = (3, 3, 256, 256)`

the first `3` means:

**which of the three images?**

The second `3` means:

**which of the three RGB channels?**

A shape records sizes, not semantic labels.

> 🇪🇸 En `(N,C,H,W) = (3,3,256,256)`:
>
> el primer `3` responde:
>
> **¿cuál de las tres imágenes?**
>
> el segundo `3` responde:
>
> **¿cuál de los tres canales RGB?**
>
> La forma guarda tamaños, no etiquetas semánticas.

</details>

## 4.3 Reshape vs. transpose / Reshape vs. transpose

This is the most important part of the notebook.

Starting from:

`photo.shape = (512, 512, 3)`

both of these commands can produce:

`(3, 512, 512)`

### Correct axis movement

`np.transpose(photo, (2, 0, 1))`

### Same shape, wrong interpretation

`photo.reshape(3, 512, 512)`

Why?

`transpose` changes **which axis goes where**.

`reshape` changes **how the stored sequence of values is grouped**.

### Analogy / Analogía

Suppose a library has books arranged as:

**shelf × position × category**

`transpose` moves the labeled dimensions.

`reshape` is more like taking the books in their current reading order and filling a differently shaped shelving unit without respecting those original labels.

> 🇪🇸 Esta es la parte más importante del cuaderno.
>
> Tanto `transpose` como `reshape` pueden producir una forma `(3,512,512)`, pero no hacen lo mismo.
>
> `transpose` cambia **qué eje ocupa cada posición**.
>
> `reshape` cambia **cómo se agrupa la secuencia de valores almacenados**.

## Exercise 3 — code that runs and is still wrong / Ejercicio 3 — código que funciona y aun así está mal

### Predict first / Predice primero

If two arrays have the same shape:

`(3, 512, 512)`

does that guarantee that:

`array_a[0]`

and:

`array_b[0]`

represent the same colour channel?

> 🇪🇸 Si dos arreglos tienen la misma forma `(3,512,512)`, ¿eso garantiza que el primer plano de ambos represente el mismo canal de color?

In [ ]:
# TODO 5 / TAREA 5
#
# EN:
# 1. Compute the correct CHW array with transpose.
# 2. Compute photo.reshape(3, 512, 512).
# 3. Verify that the shapes are equal.
# 4. Verify whether the arrays themselves are equal.
# 5. Display plane 0 from both arrays.
# 6. Explain why reshape cannot replace transpose here.
#
# ES:
# 1. Calcula el arreglo CHW correcto con transpose.
# 2. Calcula photo.reshape(3, 512, 512).
# 3. Verifica que las formas sean iguales.
# 4. Verifica si los arreglos completos son iguales.
# 5. Muestra el plano 0 de ambos.
# 6. Explica por qué reshape no puede sustituir transpose en este caso.

In [ ]:
#@title Solution / Solución — try it yourself first / inténtalo primero { display-mode: 'form' }

correct_chw = np.transpose(photo, (2, 0, 1))
wrong_reshape = photo.reshape(3, 512, 512)

print("Correct CHW / CHW correcto:", correct_chw.shape)
print("Reshape output / Salida reshape:", wrong_reshape.shape)
print()

print(
    "Same shape / Misma forma:",
    correct_chw.shape == wrong_reshape.shape,
)
print(
    "Same data arrangement / Misma organización de datos:",
    np.array_equal(correct_chw, wrong_reshape),
)

difference_fraction = np.mean(correct_chw != wrong_reshape)

print(
    f"Different positions / Posiciones diferentes: "
    f"{difference_fraction:.1%}"
)

fig, axes = plt.subplots(1, 3, figsize=(11, 3.6))

axes[0].imshow(photo)
axes[0].set_title("Original RGB / RGB original")

axes[1].imshow(correct_chw[0], cmap="gray")
axes[1].set_title(
    "transpose\nreal red channel / canal rojo real"
)

axes[2].imshow(wrong_reshape[0], cmap="gray")
axes[2].set_title(
    "reshape\nscrambled interpretation / interpretación alterada"
)

for ax in axes:
    ax.axis("off")

plt.tight_layout()
plt.show()

print()
print("EN: transpose preserved the meaning of the colour axis.")
print("ES: transpose conservó el significado del eje de color.")
print("EN: reshape reached the same shape but did not move the colour axis correctly.")
print("ES: reshape alcanzó la misma forma, pero no movió correctamente el eje de color.")

### Interactive transpose-vs-reshape explorer / Explorador interactivo transpose-vs-reshape

Choose:

- a plane `0`, `1`, or `2`;
- the method used to create `(3,512,512)`.

Compare the result with the true RGB channel.

> 🇪🇸 Elige:
>
> - un plano `0`, `1` o `2`;
> - el método usado para crear `(3,512,512)`.
>
> Compara el resultado con el canal RGB verdadero.

In [ ]:
# The correct transpose and the same-shape reshape (Exercise 3), recomputed
# here so this explorer runs whether or not the folded solution was executed.
correct_chw = np.transpose(photo, (2, 0, 1))
wrong_reshape = photo.reshape(3, 512, 512)

plane_slider = widgets.IntSlider(
    value=0,
    min=0,
    max=2,
    step=1,
    description="Plane / Plano:",
    continuous_update=False,
    style={"description_width": "100px"},
)

method_toggle = widgets.ToggleButtons(
    options=[
        ("Transpose / Transposición", "transpose"),
        ("Reshape", "reshape"),
    ],
    value="transpose",
    description="Method / Método:",
    style={"description_width": "110px"},
)

def explore_method(plane, method):
    true_channel = photo[:, :, plane]

    if method == "transpose":
        candidate = correct_chw[plane]
        method_name = "transpose / transposición"
    else:
        candidate = wrong_reshape[plane]
        method_name = "reshape"

    same = np.array_equal(candidate, true_channel)
    mae = float(
        np.mean(
            np.abs(
                candidate.astype(np.float32)
                - true_channel.astype(np.float32)
            )
        )
    )

    plt.close("all")
    fig, axes = plt.subplots(1, 2, figsize=(8.4, 3.5))

    axes[0].imshow(true_channel, cmap="gray")
    axes[0].set_title(
        f"True channel {plane} / Canal real {plane}"
    )

    axes[1].imshow(candidate, cmap="gray")
    axes[1].set_title(
        f"{method_name}\nplane/plano {plane}"
    )

    for ax in axes:
        ax.axis("off")

    plt.tight_layout()
    plt.show()

    print("Exact match / Coincidencia exacta:", same)
    print(f"Mean absolute difference / Diferencia absoluta media: {mae:.3f}")

    if same:
        print("EN: this operation preserved the intended channel semantics.")
        print("ES: esta operación conservó el significado correcto del canal.")
    else:
        print("EN: the shape is plausible, but the channel semantics are wrong.")
        print("ES: la forma parece correcta, pero el significado del canal es incorrecto.")

method_output = widgets.interactive_output(
    explore_method,
    {
        "plane": plane_slider,
        "method": method_toggle,
    },
)

display(
    widgets.VBox([
        widgets.HBox([plane_slider, method_toggle]),
        method_output,
    ])
)

## When should I use each one? / ¿Cuándo debo usar cada uno?

### Use `transpose`, `moveaxis`, or similar axis-moving tools when:

you want to change:

`HWC → CHW`

or:

`NHWC → NCHW`

because the **semantic axis positions must change**.

### Use `reshape` when:

you want to regroup dimensions while preserving the intended reading order.

For example, flattening an image:

`(H, W) → (H×W,)`

can be a valid reshape when that is the representation you intentionally want.

### Simple rule / Regla sencilla

> **Move axes → transpose. Group dimensions → reshape.**

> 🇪🇸
>
> **Mover ejes → transpose. Agrupar dimensiones → reshape.**

## Quick reasoning challenge / Reto rápido de razonamiento

For each task, decide whether you need **transpose** or **reshape**.

1. `(512,512,3) → (3,512,512)` because a model expects colour first.
2. `(8,8) → (64,)` because you want one feature vector from one digit image.
3. `(3,256,256,3) → (3,3,256,256)` because the framework expects channel before height.
4. `(1797,8,8) → (1797,64)` because each image should become one row.

> 🇪🇸 Decide si cada tarea necesita **transpose** o **reshape**.
>
> La pregunta no es solamente “¿qué forma quiero?”, sino:
>
> **“¿quiero mover ejes o agrupar dimensiones?”**

In [ ]:
reasoning_task = widgets.Dropdown(
    options=[
        ("1 · HWC → CHW", 1),
        ("2 · 8×8 image → 64 values / imagen 8×8 → 64 valores", 2),
        ("3 · NHWC → NCHW", 3),
        ("4 · (1797,8,8) → (1797,64)", 4),
    ],
    value=1,
    description="Task / Tarea:",
    style={"description_width": "100px"},
)

def explain_choice(task):
    answers = {
        1: (
            "transpose",
            "colour must move from the last axis to the first",
            "el color debe moverse del último eje al primero",
        ),
        2: (
            "reshape",
            "height and width are intentionally grouped into one feature axis",
            "alto y ancho se agrupan intencionalmente en un eje de características",
        ),
        3: (
            "transpose",
            "the colour axis must move before the spatial axes",
            "el eje de color debe moverse antes de los ejes espaciales",
        ),
        4: (
            "reshape",
            "each 8×8 image is intentionally flattened into 64 values",
            "cada imagen 8×8 se aplana intencionalmente en 64 valores",
        ),
    }

    operation, en, es = answers[task]

    print("Operation / Operación:", operation)
    print("EN:", en)
    print("ES:", es)

reasoning_output = widgets.interactive_output(
    explain_choice,
    {"task": reasoning_task},
)

display(widgets.VBox([reasoning_task, reasoning_output]))

## What just happened / Qué acaba de pasar

You worked entirely with real measured image values.

### Five ideas to remember / Cinco ideas para recordar

1. **Shape is not semantics.**  
   Two axes can both have size `3` and represent different things.

2. **HWC and CHW contain the same image values in different axis orders.**

3. **NHWC and NCHW contain the same batch values in different axis orders.**

4. **Transpose moves axes.**

5. **Reshape regroups stored values.**  
   It can produce the desired shape while giving the wrong interpretation.

### Central rule / Regla central

> **Use axis-moving operations when axis meaning changes. Use reshape when you intentionally regroup dimensions without pretending that an axis moved.**

> 🇪🇸
>
> **Usa operaciones que mueven ejes cuando cambia la posición semántica de los ejes. Usa reshape cuando quieres reagrupar dimensiones intencionalmente sin fingir que un eje cambió de lugar.**

### Final self-check / Autoevaluación final

If a program prints:

`shape = (3, 512, 512)`

is that enough information to prove that axis 0 is RGB colour?

**No. You must know how the tensor was created.**

> 🇪🇸 Si un programa imprime `(3,512,512)`, ¿es suficiente para demostrar que el eje 0 representa RGB?
>
> **No. Debes saber cómo se creó el tensor.**

---

## Time for Kahoot 🎯 / Hora de Kahoot 🎯

**Kahoot 1 — Tensor Vocabulary & Shapes / Vocabulario de tensores y formas**  
6 questions / 6 preguntas · about 5 minutes / unos 5 minutos.

Join at **kahoot.it** with the PIN on the facilitator's screen.

> 🇪🇸 Entra a **kahoot.it** con el PIN que aparece en la pantalla del facilitador.

- [Quiz details and facilitator notes](https://project-delphi.github.io/tensors-workshop/kahoot.html#quiz-1)
- [Import file (`.xlsx`)](https://github.com/project-delphi/tensors-workshop/blob/main/kahoot/kahoot_quiz_1_vocabulary_shapes.xlsx)

Next / Siguiente: **05 · Video pipeline design / Diseño de un pipeline de vídeo** — [open in Colab](https://colab.research.google.com/github/project-delphi/tensors-workshop/blob/main/notebooks/05-video-pipeline-design.ipynb).

[← Workshop site / Sitio del taller](https://project-delphi.github.io/tensors-workshop/) · [All notebooks / Todos los notebooks](https://project-delphi.github.io/tensors-workshop/notebooks.html) · [Handbook / Manual](https://project-delphi.github.io/tensors-workshop/tensors_workshop_plan_with_quizzes.html)